# generator-project-and-reshape — worked example 2: Project and reshape with einops Rearrange

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `generator-project-and-reshape`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

DCGAN generators often express the flatten-to-spatial step as `einops.rearrange('b (c h w) -> b c h w')` instead of `view`, because the named pattern documents which axis is channels versus spatial. Functionally it is the same reshape, but the explicit factorization guards against accidental axis swaps.

## Worked solution

We do the same projection but reshape with einops.

1. Project: `flat = z @ weight.T + bias`, shape `(B, C*H*W)`.
2. Reshape with `rearrange(flat, 'b (c h w) -> b c h w', c=C, h=H, w=W)`. The parentheses on the input side tell einops how to split the single flat axis: first `c` varies slowest, then `h`, then `w` fastest — matching row-major `view` order.
3. Supplying `c`, `h`, `w` lets einops check the factorization is consistent (it errors if `C*H*W` does not equal the flat size), which catches shape bugs `view` would silently accept.
4. We verify the einops result is bit-identical to a plain `view` to confirm the ordering convention matches.

In [ ]:
import torch as t
import einops
from einops import rearrange

t.manual_seed(1)
latent_dim, C, H, W = 64, 8, 4, 4
z = t.randn(3, latent_dim)
weight = t.randn(C * H * W, latent_dim) * 0.02
bias = t.zeros(C * H * W)

def project_rearrange(z, weight, bias, C, H, W):
    flat = z @ weight.T + bias
    return rearrange(flat, 'b (c h w) -> b c h w', c=C, h=H, w=W)

seed = project_rearrange(z, weight, bias, C, H, W)
plain = (z @ weight.T + bias).view(3, C, H, W)
print(seed.shape)
print('matches view:', bool(t.equal(seed, plain)))